In [12]:
import torch

%load_ext autoreload
%autoreload 2

raw_path = "/home/panda/rf_data/dataset/raw"
webdataset_path = "/home/panda/rf_data/dataset/webdataset"
samples_idx_path = "/home/panda/rf_data/dataset/samples_idx"

#nz = 1024
nz = 2048
nx = 256
#new_Ns = 4096
new_Ns = 2800

from deep_bf.data_handler import DataLoader
dl = DataLoader("/home/panda/rf_data/")

seed = 42
ratio = 0.9
transform = False
order = "CWH"
metadata = { "seed": seed, "ratio": ratio, "transform": transform, "order": order}

device = "cuda"
dtype = torch.float32

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
df = dl.get_df()
df = df.query("RF == 1 and nc == 128")
df = df[df["name"].str[:3] != "JHU"]
#df = df[df["name"] != "OSL010"]
#df = df[df["name"].str[:3] != "UFL"]

df2 = df.sort_values("name")
df2 = df2.query("RF == 1 and nc == 128")

# AREGLAR LAS KEYS YA QUE SE CAMBIARON EN DATA.CSV
keys = ["na", "fs", "aperture_width", "element_width", "pitch", "nc", "zlims", "fc"]
group = df2.groupby(keys)

In [7]:
import h5py
import hdf5plugin
from pathlib import Path
import shutil
from deep_bf.beamformers import compute_samples_idx_by_angle

beamformer = "MVB"
bfs = [beamformer]

p = Path(samples_idx_path)
shutil.rmtree(p, ignore_errors=True)
p.mkdir(parents=True, exist_ok=True)

id = 0
for group_name, mini_df in group:
    name = mini_df["name"].iloc[0]

    pw = dl.get_defined_pwdata(name, "RF")
    angle = pw.n_angles // 2

    samples_idx = compute_samples_idx_by_angle(pw, angle, nz, nx).cpu()

    with h5py.File(f"{samples_idx_path}/{id}.hdf5", "w") as f:
        f.create_dataset("samples_idx", data=samples_idx, chunks=True, **hdf5plugin.Bitshuffle(nelems=0, cname="zstd"))

    id += 1

In [14]:
for group_name, mini_df in group:
    gn = mini_df["name"]
    print(gn.iloc[0], gn.iloc[-1], gn.count())

TSH002 TSH501 500
OSL010 OSL010 1
carotid_cross_expe_dataset_rf resolution_distorsion_simu_dataset_rf 6
INS001 INS013 11
INS004 INS006 2
OSL002 OSL002 1
OSL003 OSL003 1
OSL004 OSL004 1
OSL005 OSL005 1
OSL006 OSL006 1
OSL007 OSL007 1
MYO001 MYO006 6
INS014 INS026 13
UFL001 UFL005 5


In [15]:
from deep_bf.dataset import split_webdataset, defined_webdataset
from global_variables import TRAIN_FILES_TENSORFLOW, VAL_FILES_TENSORFLOW

filter = {
    "contrast_speckle_expe_dataset_rf",
    "contrast_speckle_simu_dataset_rf",
    "resolution_distorsion_expe_dataset_rf",
    "resolution_distorsion_simu_dataset_rf",
    "carotid_cross_expe_dataset_rf",
    "carotid_long_expe_dataset_rf"
}
#filter = {}
#split_webdataset(raw_path, webdataset_path, samples_idx_path, metadata, filter=filter)
defined_webdataset(TRAIN_FILES_TENSORFLOW, VAL_FILES_TENSORFLOW, raw_path, webdataset_path, samples_idx_path, metadata)

loading gsi
gsi loaded
# writing /home/panda/rf_data/dataset/webdataset/train/dataset-000.tar 0 0.0 GB 0
# writing /home/panda/rf_data/dataset/webdataset/train/dataset-001.tar 100 0.4 GB 100
# writing /home/panda/rf_data/dataset/webdataset/train/dataset-002.tar 100 0.4 GB 200
# writing /home/panda/rf_data/dataset/webdataset/train/dataset-003.tar 100 0.4 GB 300
# writing /home/panda/rf_data/dataset/webdataset/train/dataset-004.tar 100 0.4 GB 400
# writing /home/panda/rf_data/dataset/webdataset/val/dataset-000.tar 0 0.0 GB 0


In [ ]:
# TODO: agregar el padding a la hora de crear el webdataset para tener un raw mas varaible.

if pw.n_samples <= new_Ns:
        pw.data = np.pad(pw.data, pad_width=((0,0), (0,0), (0, new_Ns - pw.n_samples)), mode='constant', constant_values=0)
    else:
        pw.data = pw.data[:, :, :new_Ns]
        pw.img_depth = (new_Ns / pw.fs) * (pw.c0 / 2)
    pw.n_samples = new_Ns